In [0]:
def _get_param(name, default=""):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG = _get_param("catalog", "fk_ce_2026_07_1382")
TASK_KEY = _get_param("task_key", "all")
JOB_RUN_ID = _get_param("job_run_id", "manual")
CHECKPOINT_NAME = _get_param("checkpoint_name", "ADHOC")

def _audit(schema_table):
    schema, table = schema_table.split(".")
    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {CATALOG}.{schema}.{table}").collect()[0][0]
    spark.sql(f"INSERT INTO {CATALOG}.ops.load_audit VALUES ('{JOB_RUN_ID}', '{TASK_KEY}', '{schema_table}', {cnt}, current_timestamp())")

In [0]:
stores_df = spark.read.table("fk_ce_2026_07_1382.silver.stores")
stores_df.write.mode("overwrite").saveAsTable("fk_ce_2026_07_1382.gold.dim_store")

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.gold.dim_product AS
SELECT
    product_id,
    product_name,
    brand,
    is_active,
    mrp,
    selling_price,
    snapshot_date,
    category_l1,
    category_l2,
    pack_size,
    pack_unit
FROM fk_ce_2026_07_1382.silver.products

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.gold.dim_customers AS
SELECT
  ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
  *
FROM fk_ce_2026_07_1382.silver.customers

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.gold.dim_date AS
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    CAST(d AS DATE) AS calendar_date,
    date_format(d, 'EEEE') AS day_name,
    EXTRACT(WEEK FROM d) AS iso_week,
    CASE WHEN date_format(d, 'E') IN ('Sat', 'Sun') THEN TRUE ELSE FALSE END AS is_weekend
FROM (
    SELECT explode(sequence(to_date('2026-03-01'), to_date('2026-03-31'), interval 1 day)) AS d
) t

In [0]:
df_orders = spark.read.table("fk_ce_2026_07_1382.silver.orders")

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.gold.fact_orders AS
SELECT
    o.order_id,
    d.date_key AS order_date_key,
    c.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    SUM(oi.quantity) AS units,
    SUM(oi.gross_amount) AS gross_amount,
    SUM(oi.discount_amount) AS discount_amount,
    SUM(oi.net_amount) AS net_item_amount,
    o.delivery_fee,
    (SUM(oi.net_amount) + o.delivery_fee) AS order_total,
    o.promised_minutes,
    o.actual_minutes,
    CASE
        WHEN o.order_status = 'DELIVERED' THEN
            CASE WHEN o.actual_minutes <= o.promised_minutes THEN TRUE ELSE FALSE END
        ELSE NULL
    END AS is_on_time
FROM fk_ce_2026_07_1382.silver.orders o
LEFT JOIN fk_ce_2026_07_1382.gold.dim_date d
    ON o.order_date = d.calendar_date
LEFT JOIN fk_ce_2026_07_1382.gold.dim_customers c
    ON o.customer_id = c.customer_id
LEFT JOIN fk_ce_2026_07_1382.gold.dim_store s
    ON o.store_id = s.store_id
LEFT JOIN fk_ce_2026_07_1382.gold.dim_product p
    ON o.product_id = p.product_id
LEFT JOIN fk_ce_2026_07_1382.silver.order_items oi 
    ON o.order_id = oi.order_id
GROUP BY
    o.order_id,
    d.date_key,
    c.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    o.delivery_fee,
    o.promised_minutes,
    o.actual_minutes

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.gold.fact_order_line AS
SELECT
    o.order_id,
    d.date_key         AS order_date_key,
    c.customer_sk,
    o.store_id,
    oi.product_id,
    o.order_status,
    oi.quantity,
    oi.unit_price,
    oi.gross_amount,
    oi.discount_amount,
    oi.net_amount
FROM fk_ce_2026_07_1382.silver.orders o
INNER JOIN fk_ce_2026_07_1382.silver.order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN fk_ce_2026_07_1382.gold.dim_date d
    ON o.order_date = d.calendar_date
LEFT JOIN fk_ce_2026_07_1382.gold.dim_customers c
    ON o.customer_id = c.customer_id

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.gold.agg_daily_store_sales AS
WITH date_range AS (
    SELECT
        MIN(to_date(CAST(order_date_key AS STRING), 'yyyyMMdd')) AS min_date,
        MAX(to_date(CAST(order_date_key AS STRING), 'yyyyMMdd')) AS max_date
    FROM fk_ce_2026_07_1382.gold.fact_orders
),
all_dates AS (
    SELECT explode(sequence(
        (SELECT min_date FROM date_range),
        (SELECT max_date FROM date_range),
        interval 1 day
    )) AS order_date
),
store_day_spine AS (
    SELECT s.store_id, s.city, d.order_date
    FROM fk_ce_2026_07_1382.gold.dim_store s
    CROSS JOIN all_dates d
),
daily_sales AS (
    SELECT
        store_id,
        order_date_key,
        COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN order_id END) AS delivered_orders,
        COUNT(DISTINCT CASE WHEN order_status = 'CANCELLED' THEN order_id END) AS cancelled_orders,
        SUM(units)                            AS units_sold,
        SUM(gross_amount)                    AS gross_amount,
        SUM(discount_amount)                AS discount_amount,
        SUM(net_item_amount)                AS net_items_revenue,
        SUM(delivery_fee)                   AS delivery_fee_revenue,
        COUNT(DISTINCT CASE WHEN is_on_time = TRUE THEN order_id END) AS on_time_orders
    FROM fk_ce_2026_07_1382.gold.fact_orders
    GROUP BY store_id, order_date_key
)
SELECT
    sd.order_date,
    sd.store_id,
    sd.city,
    COALESCE(ds.delivered_orders, 0)       AS delivered_orders,
    COALESCE(ds.cancelled_orders, 0)       AS cancelled_orders,
    COALESCE(ds.units_sold, 0)             AS units_sold,
    COALESCE(ds.gross_amount, 0)           AS gross_amount,
    COALESCE(ds.discount_amount, 0)        AS discount_amount,
    COALESCE(ds.net_items_revenue, 0)      AS net_items_revenue,
    COALESCE(ds.delivery_fee_revenue, 0)    AS delivery_fee_revenue,
    COALESCE(ds.on_time_orders, 0)         AS on_time_orders,
    CASE
        WHEN COALESCE(ds.delivered_orders, 0) = 0 THEN 0.0
        ELSE ROUND(COALESCE(ds.on_time_orders, 0) * 100.0 / ds.delivered_orders, 2)
    END AS on_time_rate_pct
FROM store_day_spine sd
LEFT JOIN daily_sales ds
    ON sd.store_id = ds.store_id
   AND to_date(CAST(ds.order_date_key AS STRING), 'yyyyMMdd') = sd.order_date
ORDER BY sd.order_date, sd.store_id

In [0]:
if TASK_KEY == "gold_build":
    _audit("gold.dim_store")
    _audit("gold.dim_product")
    _audit("gold.dim_customers")
    _audit("gold.dim_date")
    _audit("gold.fact_orders")
    _audit("gold.fact_order_line")
    _audit("gold.agg_daily_store_sales")
    dbutils.notebook.exit("done")

In [0]:
%sql

CREATE OR REPLACE VIEW fk_ce_2026_07_1382.gold.v_kpi_summary AS
WITH delivered AS (
    SELECT
        customer_id,
        order_id,
        net_item_amount,
        is_on_time
    FROM fk_ce_2026_07_1382.gold.fact_orders
    WHERE order_status = 'DELIVERED'
),
cancelled AS (
    SELECT order_id
    FROM fk_ce_2026_07_1382.gold.fact_orders
    WHERE order_status = 'CANCELLED'
),
customer_stats AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS delivered_order_count,
        COUNT(DISTINCT CASE WHEN is_on_time = TRUE THEN order_id END) AS on_time_count,
        SUM(net_item_amount) AS customer_net_revenue
    FROM delivered
    GROUP BY customer_id
)
SELECT
    (SELECT COUNT(DISTINCT order_id) FROM delivered)                       AS delivered_orders,
    (SELECT COUNT(DISTINCT order_id) FROM cancelled)                       AS cancelled_orders,
    (SELECT COALESCE(SUM(net_item_amount), 0) FROM delivered)              AS net_item_revenue,
    CASE
        WHEN (SELECT COUNT(DISTINCT order_id) FROM delivered) = 0 THEN 0.0
        ELSE ROUND(
            (SELECT COALESCE(SUM(net_item_amount), 0) FROM delivered)
            / (SELECT COUNT(DISTINCT order_id) FROM delivered),
            2
        )
    END                                                                     AS avg_order_value,
    CASE
        WHEN (SELECT COUNT(DISTINCT order_id) FROM delivered) = 0 THEN 0.0
        ELSE ROUND(
            100.0 * (SELECT COUNT(DISTINCT order_id) FROM delivered WHERE is_on_time = TRUE)
            / (SELECT COUNT(DISTINCT order_id) FROM delivered),
            2
        )
    END                                                                     AS on_time_rate_pct,
    CASE
        WHEN (SELECT COUNT(DISTINCT order_id) FROM delivered)
       + (SELECT COUNT(DISTINCT order_id) FROM cancelled) = 0 THEN 0.0
        ELSE ROUND(
            100.0 * (SELECT COUNT(DISTINCT order_id) FROM cancelled)
            / ((SELECT COUNT(DISTINCT order_id) FROM delivered)
             + (SELECT COUNT(DISTINCT order_id) FROM cancelled)),
            2
        )
    END                                                                     AS cancelation_rate_pct,
    (SELECT COUNT(DISTINCT customer_id) FROM delivered)                    AS active_customers,
    CASE
        WHEN (SELECT COUNT(DISTINCT customer_id) FROM delivered) = 0 THEN 0.0
        ELSE ROUND(
            100.0 * (SELECT COUNT(DISTINCT customer_id) FROM customer_stats WHERE delivered_order_count >= 2)
            / (SELECT COUNT(DISTINCT customer_id) FROM delivered),
            2
        )
    END                                                                     AS repeat_customer_rate_pct

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.ops.reconciliation_results AS
WITH order_count_src AS (
    SELECT COUNT(DISTINCT order_id) AS source_value
    FROM fk_ce_2026_07_1382.bronze.orders
),
order_count_tgt AS (
    SELECT
        (SELECT COUNT(*) FROM fk_ce_2026_07_1382.silver.orders)
      + (SELECT COUNT(*) FROM fk_ce_2026_07_1382.silver.quarantine_orders) AS target_value
),
line_count_src AS (
    SELECT COUNT(*) AS source_value
    FROM fk_ce_2026_07_1382.silver.order_items
),
line_count_tgt AS (
    SELECT COUNT(*) AS target_value
    FROM fk_ce_2026_07_1382.gold.fact_order_line
),
net_revenue_src AS (
    SELECT COALESCE(SUM(oi.net_amount), 0) AS source_value
    FROM fk_ce_2026_07_1382.silver.order_items oi
    INNER JOIN fk_ce_2026_07_1382.silver.orders o
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'DELIVERED'
),
net_revenue_tgt AS (
    SELECT COALESCE(SUM(net_items_revenue), 0) AS target_value
    FROM fk_ce_2026_07_1382.gold.agg_daily_store_sales
),
delivered_orders_src AS (
    SELECT COUNT(*) AS source_value
    FROM fk_ce_2026_07_1382.gold.fact_orders
    WHERE order_status = 'DELIVERED'
),
delivered_orders_tgt AS (
    SELECT COALESCE(SUM(delivered_orders), 0) AS target_value
    FROM fk_ce_2026_07_1382.gold.agg_daily_store_sales
),
current_customers_src AS (
    SELECT COUNT(DISTINCT customer_id) AS source_value
    FROM fk_ce_2026_07_1382.silver.customers
),
current_customers_tgt AS (
    SELECT COUNT(*) AS target_value
    FROM fk_ce_2026_07_1382.gold.dim_customers
    WHERE is_current = TRUE
)
SELECT
    'order_count' AS reconciliation_name,
    (SELECT source_value FROM order_count_src) AS source_value,
    (SELECT target_value FROM order_count_tgt) AS target_value,
    (SELECT source_value FROM order_count_src) = (SELECT target_value FROM order_count_tgt) AS is_match
UNION ALL
SELECT
    'line_count' AS reconciliation_name,
    (SELECT source_value FROM line_count_src) AS source_value,
    (SELECT target_value FROM line_count_tgt) AS target_value,
    (SELECT source_value FROM line_count_src) = (SELECT target_value FROM line_count_tgt) AS is_match
UNION ALL
SELECT
    'net_revenue' AS reconciliation_name,
    (SELECT source_value FROM net_revenue_src) AS source_value,
    (SELECT target_value FROM net_revenue_tgt) AS target_value,
    (SELECT source_value FROM net_revenue_src) = (SELECT target_value FROM net_revenue_tgt) AS is_match
UNION ALL
SELECT
    'delivered_orders' AS reconciliation_name,
    (SELECT source_value FROM delivered_orders_src) AS source_value,
    (SELECT target_value FROM delivered_orders_tgt) AS target_value,
    (SELECT source_value FROM delivered_orders_src) = (SELECT target_value FROM delivered_orders_tgt) AS is_match
UNION ALL
SELECT
    'current_customers' AS reconciliation_name,
    (SELECT source_value FROM current_customers_src) AS source_value,
    (SELECT target_value FROM current_customers_tgt) AS target_value,
    (SELECT source_value FROM current_customers_src) = (SELECT target_value FROM current_customers_tgt) AS is_match

In [0]:
%sql
select * from fk_ce_2026_07_1382.ops.reconciliation_results

In [0]:
if TASK_KEY in ("quality_checks", "all"):
    # H2 – Checkpoint counts: one row per table, tagged with this run's checkpoint_name
    _h2_tables = [
        "bronze.orders", "bronze.customers", "bronze.products", "bronze.stores",
        "silver.orders", "silver.order_items", "silver.quarantine_orders",
        "silver.customers", "silver.products",
        "gold.dim_customers", "gold.fact_orders", "gold.fact_order_line",
        "gold.agg_daily_store_sales",
    ]
    for t in _h2_tables:
        cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {CATALOG}.{t}").collect()[0][0]
        spark.sql(f"INSERT INTO {CATALOG}.ops.checkpoint_counts VALUES ('{CHECKPOINT_NAME}', '{t}', {cnt}, current_timestamp())")
    # H3 – Audit for quality_checks
    _audit("ops.reconciliation_results")
    _audit("ops.checkpoint_counts")
    print("H2 checkpoint_counts and H3 load_audit entries written for quality_checks.")